# Comparing Activation Functions in a Fixed CNN Reconstructor

`CNNArchitectureComparison.ipynb` and `AdvancedArchitectureComparison.ipynb` both compared different network *architectures* under a fixed activation function. This notebook holds the architecture fixed instead -- the exact `ClassicCNN` baseline from `CNNArchitectureComparison.ipynb`, reused verbatim except for one added constructor argument -- and varies only the elementwise nonlinearity used inside its conv stages: `ReLU`, `LeakyReLU`, `ELU`, `GELU`, `SiLU`, `Mish`, `Sigmoid`, and an editable `Custom` activation whose `forward()` you can replace with any expression of your own (e.g. the signed-square-root placeholder below). Training/evaluation follows the exact same `AI4AO.Trainer.Trainer` procedure, same instrument, same loss, same `TrainRunNb`, as the two reference notebooks.

Every activation compared here is parameter-free (including the `Custom` placeholder), so unlike the architecture comparisons, no per-arm capacity tuning is needed -- `ClassicCNN`'s trainable parameter count is identical across every entry in the comparison, isolating the nonlinearity itself as the only controlled variable.

This notebook reuses `CNNComparison_params.py` directly (same fictional-demo, nominal-geometry modulated Pyramid WFS + DM as both reference notebooks), so all three notebooks compare on the identical instrument.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Configuration and instrument

Same fictional-demo, nominal-geometry modulated Pyramid WFS + DM as `CNNArchitectureComparison.ipynb`/`AdvancedArchitectureComparison.ipynb` -- this notebook imports `CNNComparison_params.py` directly rather than forking its own copy, so all three notebooks always compare on the identical instrument. `wfs`/`dm` are built fresh and frozen (`.eval()`); only each activation variant's `ClassicCNN` weights are trained.

In [ ]:
paramfile = 'CNNComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = "../../Data/ActivationComparison/"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")

## `ClassicCNN` with a swappable activation

The exact same `num_downsamples`/`conv_stage`/`ClassicCNN` from `CNNArchitectureComparison.ipynb`, with one change: `conv_stage` and `ClassicCNN` now take an `activation` argument (an `nn.Module` subclass, e.g. `nn.ReLU`) instead of hardcoding `nn.LeakyReLU`, so the exact same architecture -- same depth (`num_downsamples(Nout)`), same `base_channels=28`, same downsampling structure -- can be instantiated once per candidate nonlinearity below. `activation` is applied after each of the two `3x3` convs in every stage; the final `Linear` head has no activation, matching the original.

In [ ]:
def num_downsamples(size, min_size=2):
    """How many stride-2 poolings a feature map of this spatial size can
    take before dropping below min_size pixels."""
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


def conv_stage(in_channels, out_channels, activation):
    """Two 3x3 convs at out_channels, then a 2x downsampling MaxPool."""
    return [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        activation(),
        nn.MaxPool2d(2),
    ]


class ClassicCNN(nn.Module):
    """Plain, non-grouped Conv2d stack, identical to CNNArchitectureComparison.ipynb's
    ClassicCNN except that the activation used inside every conv_stage is now a
    constructor argument instead of a hardcoded nn.LeakyReLU."""

    def __init__(self, n_channels, Nmodes, Nout, activation, base_channels=28):
        super().__init__()

        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1], activation)
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)

## `CustomActivation` -- edit `forward()` to try your own expression

A single `nn.Module` with one editable elementwise `forward()` -- change the expression below and rerun the notebook to compare a new candidate against the seven standard activations, no other code needs to change.

The placeholder here is a signed square root, `sign(x) * sqrt(|x|)`, which is mathematically the same function as `sqrt(x) if x >= 0 else -sqrt(-x)` but written to avoid ever evaluating `sqrt` of a negative number: naively computing both `torch.sqrt(x)` and `-torch.sqrt(-x)` and selecting with `torch.where` still evaluates `torch.sqrt` on the *whole* tensor for both branches, so the invalid branch produces `NaN`s that can leak into the gradient of the selected branch through autograd's chain rule even though `torch.where`'s forward output looks fine. Worth keeping in mind for whatever expression you substitute here -- an occasional NaN gradient can silently derail a long training run.

In [ ]:
class CustomActivation(nn.Module):
    """Editable elementwise activation -- replace the expression in forward() with
    whatever you want to try next."""

    def forward(self, x):
        return torch.sign(x) * torch.sqrt(torch.abs(x)+1e-6)

## Instantiating all eight variants

Every activation below is parameter-free, so `ClassicCNN`'s trainable parameter count is identical across every entry -- printed below as a sanity check, not because any per-arm size tuning was needed.

In [ ]:
activation_fns = {
    "ReLU": nn.ReLU,
    "LeakyReLU": nn.LeakyReLU,
    "ELU": nn.ELU,
    "GELU": nn.GELU,
    "SiLU": nn.SiLU,
    "Mish": nn.Mish,
    "Sigmoid": nn.Sigmoid,
    "Custom": CustomActivation,
}

architectures = {
    name: ClassicCNN(n_channels, Nmodes, Nout, activation=act).to(device=device)
    for name, act in activation_fns.items()
}

for name, model in architectures.items():
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name:12s} -- total trainable parameters: {total_params:,}")

## Training

Following `CNNArchitectureComparison.ipynb`'s exact procedure: a fresh `AdamW` optimizer and a fresh `Trainer` per activation variant, all sharing the same `wfs`/`dm`/`framePreprocessor`/`dataset`/`loss` objects, each calling `trainer.train(TrainRunNb, ClosedLoopIterations)`.

Same deliberate trade-off as both reference notebooks: `Trainer.train()` redraws `self.dataset[0]` fresh at every outer step regardless of which `Trainer` instance calls it, so the eight variants train on independent i.i.d. draws from the same distribution rather than paired ones. Only the seeded closed-loop rollout further down is a true apples-to-apples comparison; the training curves are directionally informative only.

`Sigmoid` in particular is a classic vanishing-gradient risk this deep (`num_downsamples(Nout)` stages, each with two activations) -- if it trains far worse than the others, that is an expected, physically meaningful result of this comparison rather than a bug.

**A checkpoint-reuse gotcha specific to this notebook:** every activation here is parameter-free, so `ClassicCNN`'s `state_dict` shapes never change no matter what `CustomActivation.forward()` computes. That means the `RuntimeError`-on-shape-mismatch fallback below, which lets the architecture-comparison notebooks safely reuse/discard stale checkpoints after a hyperparameter edit, **will not** catch an edited `CustomActivation` expression -- if you change `forward()` and rerun, `trainer.load_checkpoint` for `"Custom"` will silently resume from weights trained under the *old* expression. Delete `Data/ActivationComparison/Custom.pth` (or the whole `PATH` folder) before rerunning after editing `CustomActivation`.

In [ ]:
# Closed-loop (BPTT) iterations per optimizer step in the second training stage. Stage 1 uses
# TrainParams['ClosedLoopIterations'] from CNNComparison_params.py (a single pass); this is the
# one training setting that is not in that shared file, so it lives here.
CLOSED_LOOP_ITERATIONS_STAGE2 = 10

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} (stage 1: single pass) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH + f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by an earlier,
        # differently-sized version of this architecture (e.g. after editing base_channels
        # above) -- old checkpoints under PATH are then no longer compatible and should be
        # deleted or ignored. It does NOT cover an edited CustomActivation expression, see
        # the markdown note above.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

    trainer.save_checkpoint(PATH + f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal

### Stage 2 -- closed-loop training

The loop above (stage 1) trains every model with `TrainParams['ClosedLoopIterations']` from `CNNComparison_params.py`, i.e. a single pass per optimizer step. The next cell repeats the identical loop with `CLOSED_LOOP_ITERATIONS_STAGE2` closed-loop iterations per optimizer step, continuing from the weights stage 1 just saved (`load_checkpoint` at the top of the loop, and a fresh `AdamW` per model, as in stage 1).

The checkpoint under `PATH` is overwritten with the stage-2 weights, and from here on `trainers`/`loss_trackers`/`loss_trackers_ideal` -- the ones the plotting and evaluation cells below use -- refer to stage 2.

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in architectures.items():
    print(f"=== Training {name} (stage 2: closed loop) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH + f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by an earlier,
        # differently-sized version of this architecture (e.g. after editing base_channels
        # above) -- old checkpoints under PATH are then no longer compatible and should be
        # deleted or ignored. It does NOT cover an edited CustomActivation expression, see
        # the markdown note above.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], CLOSED_LOOP_ITERATIONS_STAGE2)

    trainer.save_checkpoint(PATH + f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal

## Comparing training-loss curves

`Trainer.plot_losses` only plots one tracker pair at a time, so we reimplement just its smoothing-and-overlay logic here for all eight trackers at once. The oracle "ideal loss" bound (`z_inv` applied to the true residual OPD) is a property of the atmosphere/noise distribution and the DM, not of the activation function, so the eight `loss_tracker_ideal` curves are statistically equivalent -- only one is shown, as a reference lower bound shared by all eight.

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

fig, ax = plt.subplots(figsize=(8, 5))
for name, tracker in loss_trackers.items():
    ax.plot(smooth(tracker), label=name)

first_ideal = next(iter(loss_trackers_ideal.values()))
ax.plot(smooth(first_ideal), label="Oracle bound (ideal)", color="black", linestyle="--")

ax.set_xlabel("Iteration")
ax.set_ylabel(r"Loss")
ax.legend()
plt.show()

## Comparing closed-loop residual wavefront error (nm RMS)

For each activation variant, reseed immediately before the rollout so all eight see identical wind/r0/noise draws, then run `trainer.evaluate()` and compute the steady-state residual wavefront error in nm RMS over the pupil, excluding the leaky-integrator warm-up (`i > n_steps * 0.3`, the same convention `Trainer.evaluate()` and both reference notebooks use).

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator warm-up
    period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    print(f"{name:12s} -- steady-state residual: {nm_rms[name]:.1f} nm RMS")

In [ ]:
names = list(nm_rms.keys())
rms_values = [nm_rms[n] for n in names]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(names, rms_values)
ax.set_ylabel("Steady-state residual (nm RMS)")
ax.set_title("Closed-loop performance by activation function")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Visualizing the best activation's closed loop

A closer look at whichever activation came out on top above -- the same rollout-animation pattern `CNNArchitectureComparison.ipynb`/`05_TrainingAReconstructor.ipynb` use.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best activation: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

n_frames = 60
torch.manual_seed(TrainParams['Seed'])
result = trainers[best_name].evaluate(n_steps=n_frames, dataset=eval_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
        {"tensor": torch.sqrt(result.psfs[0]), "title": "PSF", "same_scale": True},
    ],
    max_channel_number=9
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
            {"tensor": torch.sqrt(result.psfs[i]), "title": "PSF", "same_scale": True},
        ],
        fig=fig, axes=axes,
        max_channel_number=9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())